In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.dummy import DummyClassifier
from sklearn.svm import LinearSVC


In [2]:
DATA_DIR = "/kaggle/input/plantvillage-dataset/color"
TARGET_SIZE = (64, 64)   # as given in pseudocode
RANDOM_STATE = 42

In [3]:
def load_data(data_dir, target_size):
    X = []
    y = []

    class_dirs = sorted([p for p in Path(data_dir).iterdir() if p.is_dir()])

    for class_dir in class_dirs:
        label = class_dir.name

        for img_path in class_dir.iterdir():
            try:
                img = Image.open(img_path).convert("RGB")
                img = img.resize(target_size)
                img_array = np.array(img).flatten()   # CRITICAL STEP
                X.append(img_array)
                y.append(label)
            except:
                continue

    return np.array(X), np.array(y)


In [4]:
X, y = load_data(DATA_DIR, TARGET_SIZE)

print("Feature matrix shape:", X.shape)
print("Labels shape:", y.shape)


Feature matrix shape: (54305, 12288)
Labels shape: (54305,)


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)


In [6]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [7]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

dummy_preds = dummy.predict(X_test)
dummy_acc = accuracy_score(y_test, dummy_preds)

print("Dummy Baseline Accuracy:", dummy_acc)


Dummy Baseline Accuracy: 0.1014639535954332


In [8]:
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

svm = SGDClassifier(
    loss="hinge",        # Linear SVM
    penalty="l2",
    alpha=1e-4,          # Regularization (important for accuracy)
    max_iter=50,         # More iterations = better accuracy
    tol=1e-3,
    learning_rate="optimal",
    class_weight="balanced",  # handles class imbalance (very important)
    random_state=42,
    n_jobs=-1,
    verbose=1
)

svm.fit(X_train, y_train)

svm_preds = svm.predict(X_test)

svm_acc = accuracy_score(y_test, svm_preds)
print("SVM Accuracy:", svm_acc)

print("\nClassification Report:")
print(classification_report(y_test, svm_preds))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, svm_preds))


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.


-- Epoch 1
-- Epoch 1
-- Epoch 1
-- Epoch 1
Norm: 516.68, NNZs: 12288, Bias: -7620.389546, T: 43444, Avg. loss: 948.218724
Total training time: 1.15 seconds.
-- Epoch 2
Norm: 743.22, NNZs: 12288, Bias: -7869.276061, T: 43444, Avg. loss: 895.167941
Total training time: 1.20 seconds.
-- Epoch 2
Norm: 656.27, NNZs: 12288, Bias: -7702.165143, T: 43444, Avg. loss: 858.603832
Total training time: 1.21 seconds.
-- Epoch 2
Norm: 766.84, NNZs: 12288, Bias: -7861.327389, T: 43444, Avg. loss: 1022.310907
Total training time: 1.21 seconds.
-- Epoch 2
Norm: 414.99, NNZs: 12288, Bias: -7588.935650, T: 86888, Avg. loss: 57.129609
Total training time: 2.17 seconds.
-- Epoch 3
Norm: 646.37, NNZs: 12288, Bias: -7831.989314, T: 86888, Avg. loss: 23.704480
Total training time: 2.20 seconds.
-- Epoch 3
Norm: 527.17, NNZs: 12288, Bias: -7685.028087, T: 86888, Avg. loss: 14.634606
Total training time: 2.20 seconds.
-- Epoch 3
Norm: 675.28, NNZs: 12288, Bias: -7798.133088, T: 86888, Avg. loss: 44.636531
Total

[Parallel(n_jobs=-1)]: Done  38 out of  38 | elapsed:  7.4min finished
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_stochastic_gradient.py:738: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


SVM Accuracy: 0.6731424362397569

Classification Report:
                                                    precision    recall  f1-score   support

                                Apple___Apple_scab       0.51      0.62      0.56       126
                                 Apple___Black_rot       0.55      0.60      0.57       124
                          Apple___Cedar_apple_rust       0.11      0.42      0.17        55
                                   Apple___healthy       0.55      0.45      0.49       329
                               Blueberry___healthy       0.70      0.62      0.66       300
          Cherry_(including_sour)___Powdery_mildew       0.43      0.61      0.50       210
                 Cherry_(including_sour)___healthy       0.67      0.74      0.70       171
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot       0.46      0.51      0.49       103
                       Corn_(maize)___Common_rust_       0.97      0.85      0.91       239
               Corn_(m